# 04 — dbt-style Marts (Serving Layer)

This notebook builds **consumption-ready marts** from Silver/Gold.
Goals:
- Stable schemas for BI / dashboards
- Deterministic outputs
- Validations and row-count reconciliation
- Delta write hygiene (overwrite + physical cleanup in local/dev)

In [1]:
DATASET = "nyc_tlc_yellow"

SILVER_PATH = f"../lakehouse/silver/{DATASET}"
GOLD_KPI_DAILY_PATH = f"../lakehouse/gold/{DATASET}/kpi_daily_overview"

MART_PATH = f"../lakehouse/marts/{DATASET}"

YEAR = 2024
MONTH = 1

MART_DAILY_REVENUE = f"{MART_PATH}/mart_daily_revenue"
FACT_TRIPS = f"{MART_PATH}/fact_trips"
DIM_DATE = f"{MART_PATH}/dim_date"

print("SILVER_PATH:", SILVER_PATH)
print("GOLD_KPI_DAILY_PATH:", GOLD_KPI_DAILY_PATH)
print("MART_PATH:", MART_PATH)
print("YEAR/MONTH:", YEAR, MONTH)

SILVER_PATH: ../lakehouse/silver/nyc_tlc_yellow
GOLD_KPI_DAILY_PATH: ../lakehouse/gold/nyc_tlc_yellow/kpi_daily_overview
MART_PATH: ../lakehouse/marts/nyc_tlc_yellow
YEAR/MONTH: 2024 1


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("lakehouse-marts")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

26/02/24 14:54:47 WARN Utils: Your hostname, willian-A320M-S2H resolves to a loopback address: 127.0.1.1; using 192.168.0.110 instead (on interface enp6s0)
26/02/24 14:54:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/willian/PhaifferTech/nyc-tlc-lakehouse/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/willian/.ivy2/cache
The jars for the packages stored in: /home/willian/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6051b3df-87e1-4346-b112-2cfe18051acc;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 233ms :: artifacts dl 8ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   

In [3]:
from pyspark.sql import functions as F

silver_df = spark.read.format("delta").load(SILVER_PATH)
gold_kpi_df = spark.read.format("delta").load(GOLD_KPI_DAILY_PATH)

print("[Silver] rows:", silver_df.count())
print("[Gold KPI] days:", gold_kpi_df.count())

silver_bounds = silver_df.select(
    F.min("pickup_date").alias("min_pickup_date"),
    F.max("pickup_date").alias("max_pickup_date"),
).collect()[0]

print("[Silver] pickup_date min:", silver_bounds["min_pickup_date"])
print("[Silver] pickup_date max:", silver_bounds["max_pickup_date"])

26/02/24 14:54:57 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


[Silver] rows: 2839382


[Gold KPI] days: 31
[Silver] pickup_date min: 2024-01-01
[Silver] pickup_date max: 2024-01-31


In [4]:
# ==============================
# MART 1: mart_daily_revenue (from Gold KPI)
# ==============================

mart_daily_revenue_df = (
    gold_kpi_df
    .select(
        "pickup_date",
        "trips",
        "total_fare",
        "total_tip",
        "total_revenue",
        "avg_trip_distance",
        "avg_trip_duration_min",
        "avg_passenger_count",
    )
    # Add partition-friendly columns deterministically (do not assume they exist in Gold)
    .withColumn("year", F.year(F.col("pickup_date")))
    .withColumn("month", F.month(F.col("pickup_date")))
    .orderBy(F.col("pickup_date").asc())
)

print("[mart_daily_revenue] rows (days):", mart_daily_revenue_df.count())
mart_daily_revenue_df.show(31, truncate=False)

[mart_daily_revenue] rows (days): 31


26/02/24 14:55:07 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


+-----------+------+------------------+------------------+------------------+-----------------+---------------------+-------------------+----+-----+
|pickup_date|trips |total_fare        |total_tip         |total_revenue     |avg_trip_distance|avg_trip_duration_min|avg_passenger_count|year|month|
+-----------+------+------------------+------------------+------------------+-----------------+---------------------+-------------------+----+-----+
|2024-01-01 |74830 |1662841.4800000603|256689.6099999985 |2324594.019999933 |4.665            |16.689               |1.568              |2024|1    |
|2024-01-02 |72341 |1548093.8100000108|259731.07999999783|2255838.3599998797|4.218            |17.063               |1.432              |2024|1    |
|2024-01-03 |79188 |1590700.8200000047|271635.90999999957|2341511.009999927 |3.956            |16.677               |1.395              |2024|1    |
|2024-01-04 |99066 |1860543.840000016 |332253.1699999981 |2787599.5399999293|3.367            |16.175     

In [5]:
import os
import shutil

# Ensure deterministic physical cleanup for local/dev environments
if os.path.exists(MART_DAILY_REVENUE):
    shutil.rmtree(MART_DAILY_REVENUE)

(
    mart_daily_revenue_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(MART_DAILY_REVENUE)
)

print("mart_daily_revenue written to:", MART_DAILY_REVENUE)

mart_daily_revenue written to: ../lakehouse/marts/nyc_tlc_yellow/mart_daily_revenue


In [6]:
mart_daily_revenue_check = spark.read.format("delta").load(MART_DAILY_REVENUE)

print("[mart_daily_revenue] reload rows:", mart_daily_revenue_check.count())

bounds = mart_daily_revenue_check.select(
    F.min("pickup_date").alias("min_pickup_date"),
    F.max("pickup_date").alias("max_pickup_date"),
).collect()[0]

print("[mart_daily_revenue] min pickup_date:", bounds["min_pickup_date"])
print("[mart_daily_revenue] max pickup_date:", bounds["max_pickup_date"])

assert 28 <= mart_daily_revenue_check.count() <= 31, "Unexpected number of days in mart_daily_revenue."

[mart_daily_revenue] reload rows: 31
[mart_daily_revenue] min pickup_date: 2024-01-01
[mart_daily_revenue] max pickup_date: 2024-01-31


In [7]:
# ==============================
# FACT: fact_trips (from Silver)
# ==============================

fact_trips_df = (
    silver_df
    .withColumn(
        "trip_duration_sec",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")).cast("long")
    )
    .withColumn("trip_duration_min", (F.col("trip_duration_sec") / F.lit(60.0)).cast("double"))
    .select(
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "pickup_date",
        "passenger_count",
        "trip_distance",
        "fare_amount",
        "tip_amount",
        "tolls_amount",
        "trip_duration_sec",
        "trip_duration_min",
    )
)

print("[fact_trips] rows:", fact_trips_df.count())

stats = fact_trips_df.select(
    F.min("pickup_date").alias("min_pickup_date"),
    F.max("pickup_date").alias("max_pickup_date"),
    F.min("trip_duration_sec").alias("min_duration_sec"),
    F.max("trip_duration_sec").alias("max_duration_sec"),
).collect()[0]

print("[fact_trips] min pickup_date:", stats["min_pickup_date"])
print("[fact_trips] max pickup_date:", stats["max_pickup_date"])
print("[fact_trips] min duration sec:", stats["min_duration_sec"])
print("[fact_trips] max duration sec:", stats["max_duration_sec"])

[fact_trips] rows: 2839382
[fact_trips] min pickup_date: 2024-01-01
[fact_trips] max pickup_date: 2024-01-31
[fact_trips] min duration sec: 1
[fact_trips] max duration sec: 567324


In [8]:
import os
import shutil

if os.path.exists(FACT_TRIPS):
    shutil.rmtree(FACT_TRIPS)

(
    fact_trips_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("pickup_date")
    .save(FACT_TRIPS)
)

print("fact_trips written to:", FACT_TRIPS)

fact_trips written to: ../lakehouse/marts/nyc_tlc_yellow/fact_trips


In [9]:
fact_trips_check = spark.read.format("delta").load(FACT_TRIPS)

print("[fact_trips] reload rows:", fact_trips_check.count())

bounds = fact_trips_check.select(
    F.min("pickup_date").alias("min_pickup_date"),
    F.max("pickup_date").alias("max_pickup_date"),
).collect()[0]

print("[fact_trips] min pickup_date:", bounds["min_pickup_date"])
print("[fact_trips] max pickup_date:", bounds["max_pickup_date"])

assert fact_trips_check.count() == silver_df.count(), "fact_trips row count must match Silver row count."

[fact_trips] reload rows: 2839382
[fact_trips] min pickup_date: 2024-01-01
[fact_trips] max pickup_date: 2024-01-31


In [10]:
# ==============================
# DIM: dim_date (BI-ready calendar dimension)
# ==============================

from pyspark.sql import functions as F
from datetime import date

# Use Python date for deterministic base (avoids Spark date_add type issues)
base_date = date(YEAR, MONTH, 1)

start_date = F.lit(base_date)
end_date = F.add_months(start_date, 1)

days_in_month = (
    spark.range(0, 32)
    .withColumn("date", F.date_add(start_date, F.col("id").cast("int")))
    .filter((F.col("date") >= start_date) & (F.col("date") < end_date))
    .select("date")
)

dim_date_df = (
    days_in_month
    .withColumn("date_key", F.date_format(F.col("date"), "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("day_of_week", F.dayofweek("date"))
    .withColumn("day_name", F.date_format(F.col("date"), "EEEE"))
    .withColumn("month_name", F.date_format(F.col("date"), "MMMM"))
    .withColumn("is_weekend", F.col("day_of_week").isin([1, 7]).cast("boolean"))
    .orderBy(F.col("date").asc())
)

print("[dim_date] rows:", dim_date_df.count())
dim_date_df.show(31, truncate=False)

[dim_date] rows: 31
+----------+--------+----+-----+---+-----------+---------+----------+----------+
|date      |date_key|year|month|day|day_of_week|day_name |month_name|is_weekend|
+----------+--------+----+-----+---+-----------+---------+----------+----------+
|2024-01-01|20240101|2024|1    |1  |2          |Monday   |January   |false     |
|2024-01-02|20240102|2024|1    |2  |3          |Tuesday  |January   |false     |
|2024-01-03|20240103|2024|1    |3  |4          |Wednesday|January   |false     |
|2024-01-04|20240104|2024|1    |4  |5          |Thursday |January   |false     |
|2024-01-05|20240105|2024|1    |5  |6          |Friday   |January   |false     |
|2024-01-06|20240106|2024|1    |6  |7          |Saturday |January   |true      |
|2024-01-07|20240107|2024|1    |7  |1          |Sunday   |January   |true      |
|2024-01-08|20240108|2024|1    |8  |2          |Monday   |January   |false     |
|2024-01-09|20240109|2024|1    |9  |3          |Tuesday  |January   |false     |
|2024-01

In [11]:
import os
import shutil

# Ensure deterministic physical cleanup for local/dev environments
if os.path.exists(DIM_DATE):
    shutil.rmtree(DIM_DATE)

(
    dim_date_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(DIM_DATE)
)

print("dim_date written to:", DIM_DATE)

dim_date written to: ../lakehouse/marts/nyc_tlc_yellow/dim_date


In [12]:
dim_date_check = spark.read.format("delta").load(DIM_DATE)

print("[dim_date] reload rows:", dim_date_check.count())

bounds = dim_date_check.select(
    F.min("date").alias("min_date"),
    F.max("date").alias("max_date"),
).collect()[0]

print("[dim_date] min_date:", bounds["min_date"])
print("[dim_date] max_date:", bounds["max_date"])

# Month-length assertion (portfolio friendly)
assert 28 <= dim_date_check.count() <= 31, "Unexpected number of rows in dim_date."

[dim_date] reload rows: 31
[dim_date] min_date: 2024-01-01
[dim_date] max_date: 2024-01-31


In [13]:
# ==============================
# BI-ready keys: add date_key to fact_trips
# ==============================

fact_trips_bi_df = (
    fact_trips_check
    .withColumn("pickup_date_key", F.date_format(F.col("pickup_date"), "yyyyMMdd").cast("int"))
)

print("[fact_trips_bi] rows:", fact_trips_bi_df.count())

# Quick join check with dim_date (should match fact rows if every pickup_date exists in dim_date)
joined_rows = (
    fact_trips_bi_df
    .join(dim_date_check.select("date_key"), fact_trips_bi_df["pickup_date_key"] == dim_date_check["date_key"], "left")
    .count()
)

print("[fact_trips_bi] join rows (left join):", joined_rows)
assert joined_rows == fact_trips_bi_df.count(), "Join row count mismatch on dim_date (unexpected)."

[fact_trips_bi] rows: 2839382


[fact_trips_bi] join rows (left join): 2839382
